In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 00 · Setup — Catálogo, schemas e Volume
# MAGIC
# MAGIC **Objetivo:** preparar o ambiente do projeto *Wine Quality Lakehouse* no Unity Catalog, com um schema por camada
# MAGIC da arquitetura medalhão e um Volume para receber os arquivos brutos.
# MAGIC
# MAGIC ```
# MAGIC workspace (catálogo)
# MAGIC ├── wine_quality
# MAGIC │   └── raw_files (Volume)  <- CSVs originais da UCI
# MAGIC ├── wine_bronze             <- dados brutos em Delta
# MAGIC ├── wine_silver             <- dados tipados, limpos e validados
# MAGIC └── wine_gold               <- features e tabelas analíticas
# MAGIC ```
# MAGIC
# MAGIC Este notebook é **idempotente**: pode ser executado quantas vezes for preciso sem recriar ou apagar nada.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Configuração

# COMMAND ----------

CATALOG = "workspace"

SOURCE_SCHEMA = "wine_quality"
VOLUME = "raw_files"

LAYER_SCHEMAS = {
    "wine_bronze": "Camada Bronze: dados originais com metadados de ingestão",
    "wine_silver": "Camada Silver: dados tipados, limpos e validados",
    "wine_gold": "Camada Gold: features para modelagem e tabelas analíticas",
}

RAW_PATH = f"/Volumes/{CATALOG}/{SOURCE_SCHEMA}/{VOLUME}"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Verificar o catálogo

# COMMAND ----------

available_catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
print("Catálogos disponíveis:", available_catalogs)

assert CATALOG in available_catalogs, (
    f"Catálogo '{CATALOG}' não encontrado. "
    "Ajuste a variável CATALOG para um dos catálogos disponíveis."
)

available_schemas = [
    row[0] for row in spark.sql(f"SHOW SCHEMAS IN `{CATALOG}`").collect()
]
print("Schemas disponíveis:", available_schemas)

assert SOURCE_SCHEMA in available_schemas, (
    f"Schema de origem '{SOURCE_SCHEMA}' não encontrado em '{CATALOG}'. "
    "Confira onde os arquivos CSV foram enviados."
)

spark.sql(f"USE CATALOG `{CATALOG}`")
print(f"Catálogo em uso: {CATALOG}")
print(f"Schema de origem dos CSVs: {CATALOG}.{SOURCE_SCHEMA}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Criar os schemas das camadas

# COMMAND ----------

for schema, comment in LAYER_SCHEMAS.items():
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{schema}` "
        f"COMMENT '{comment}'"
    )
    print(f"OK: {CATALOG}.{schema}")    

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Criar o Volume para os arquivos brutos
# MAGIC

# COMMAND ----------

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SOURCE_SCHEMA}`.`{VOLUME}` "
    "COMMENT 'Arquivos brutos originais da UCI Wine Quality'"
)

print(f"Volume de origem: {RAW_PATH}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Conferir o que foi criado

# COMMAND ----------

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

# COMMAND ----------

display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.{SOURCE_SCHEMA}"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Upload dos arquivos brutos (manual, uma vez)

# COMMAND ----------

try:
    files = [item.name.rstrip("/") for item in dbutils.fs.ls(RAW_PATH)]
    print("Arquivos encontrados:", files)

    expected_files = {"winequality-red.csv", "winequality-white.csv"}
    missing_files = expected_files - set(files)

    if missing_files:
        print("ATENÇÃO — arquivos não encontrados:", sorted(missing_files))
        print(f"Confira se os CSVs estão em: {RAW_PATH}")
    else:
        print("OK: os dois arquivos CSV estão no Volume.")
except Exception as error:
    print("Não foi possível listar o conteúdo do Volume.")
    print(f"Caminho verificado: {RAW_PATH}")
    print(f"Tipo do erro: {type(error).__name__}")
    print(f"Mensagem: {error}")
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## Próximo passo
# MAGIC
# MAGIC `01_bronze_ingestion`: leitura dos CSVs e gravação da tabela Delta `wine_quality_raw` na camada Bronze.